In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("merge_keys","")
dbutils.widgets.text("load_type","")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
from pyspark.sql import functions as F
updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp",F.current_timestamp())


In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresync.silver.{table_name}"

if load_type == "FULL":
    # Overwrite entire target table
    updated_df.write.mode("overwrite").saveAsTable(target_table)

elif load_type == "APPEND":
    # Append new records without touching existing data
    updated_df.write.mode("append").saveAsTable(target_table)

elif load_type == "MERGE":
    keys = [k.strip() for k in merge_keys.split(",")]
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])

    if DeltaTable.isDeltaTable(spark, target_table):
        delta_table = DeltaTable.forName(spark, target_table)
        # Update all columns except insert_timestamp to preserve original record creation time
        update_cols = {c: f"source.{c}" for c in updated_df.columns if c != "insert_timestamp"}
        (
            delta_table.alias("target")
            .merge(updated_df.alias("source"), merge_condition)
            .whenMatchedUpdate(set=update_cols)
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        # Target table doesn't exist yet — do initial full load
        updated_df.write.mode("overwrite").saveAsTable(target_table)

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected FULL, APPEND, or MERGE.")

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))